## Load labeled data
Load positive and negative sentence sets saved during the labeling phase.

In [1]:
import pandas as pd
import os
os.chdir("..")

positives = pd.read_parquet("data/raw/positives.parquet")
true_negatives = pd.read_parquet("data/raw/true_negatives.parquet")
mass_negatives = pd.read_parquet("data/raw/mass_negatives.parquet")

print(f"Positives: {len(positives)}")
print(f"True negatives: {len(true_negatives)}")
print(f"Mass negatives: {len(mass_negatives)}")

Positives: 962
True negatives: 18600
Mass negatives: 80000


## Combine and verify dataset integrity
Merge all three sets, confirm no sentence overlap across classes, and check label distribution.

In [2]:
df = pd.concat([positives, true_negatives, mass_negatives], ignore_index=True)

print(f"Total sentences: {len(df)}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")
print(f"\nImbalance ratio: {df['label'].value_counts()[0] / df['label'].value_counts()[1]:.1f}:1")
print(f"\nDuplicate sentences: {df['sentence'].duplicated().sum()}")

Total sentences: 99562

Label distribution:
label
0    98600
1      962
Name: count, dtype: int64

Imbalance ratio: 102.5:1

Duplicate sentences: 260


## Deduplication

In [3]:
df = df.drop_duplicates(subset='sentence').reset_index(drop=True)

print(f"Total after dedup: {len(df)}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")
print(f"Imbalance ratio: {df['label'].value_counts()[0] / df['label'].value_counts()[1]:.1f}:1")

Total after dedup: 99302

Label distribution:
label
0    98340
1      962
Name: count, dtype: int64
Imbalance ratio: 102.2:1


## Three-way stratified split — train / calibration / test
Splits real data only. Synthetic samples will be added to train only later.
Test and calibration are locked — real data only, never augmented.

In [5]:
from sklearn.model_selection import train_test_split

# First split off test (20%)
train_cal_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)

# Then split calibration from remaining (10% of total = ~12.5% of remainder)
train_df, cal_df = train_test_split(
    train_cal_df, test_size=0.125, random_state=42, stratify=train_cal_df['label']
)

print(f"Train:       {len(train_df)} | Positives: {train_df['label'].sum()}")
print(f"Calibration: {len(cal_df)}  | Positives: {cal_df['label'].sum()}")
print(f"Test:        {len(test_df)}  | Positives: {test_df['label'].sum()}")

train_df.to_parquet("data/processed/train.parquet", index=False)
cal_df.to_parquet("data/processed/calibration.parquet", index=False)
test_df.to_parquet("data/processed/test.parquet", index=False)
df.to_parquet("data/processed/dataset_full.parquet", index=False)
print("Saved.")

Train:       69510 | Positives: 674
Calibration: 9931  | Positives: 96
Test:        19861  | Positives: 192
Saved.
